# Descriptive Statistics & EDA with Pandas
### Housing dataset walkthrough

**What you'll learn**
1. Load and inspect a dataset
2. Central tendency: mean, median, mode
3. Spread: range, standard deviation, variance, IQR
4. Shape: skewness and kurtosis
5. Outlier detection with the 1.5 x IQR rule
6. Choosing the right plot (histogram, boxplot, scatter)

**Requirements:** `pandas`, `matplotlib`, and `housing.csv` in the same folder as this notebook.

## 1. Load and inspect the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

path = "housing.csv"
df = pd.read_csv(path)

print("Shape (rows, columns):", df.shape)
print("Dimensions:", df.ndim)
print("Total cells:", df.size)
print("\nColumns:", list(df.columns))

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.dtypes

In [ ]:
df.info()

`describe()` gives a quick summary of numeric columns. Use `include='all'` to see categorical columns too, or `exclude='number'` to see only the non-numeric ones.

In [ ]:
df.describe()

In [ ]:
df.describe(include='all')

In [ ]:
df.describe(exclude='number')

## 2. Data quality checks
Before computing statistics, confirm that each `House_ID` is unique and that nothing is missing.

In [ ]:
print("Rows:", len(df))
print("Unique House_IDs:", df["House_ID"].nunique())
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# Show any duplicated House_IDs (an empty result means all IDs are unique)
df[df["House_ID"].duplicated(keep=False)].sort_values("House_ID")

## 3. Central tendency
Where is the "centre" of each column?

- **Mean**: the arithmetic average. It is pulled by extreme values.
- **Median**: the middle value. It is robust to extreme values.
- **Mode**: the most frequent value. If several values tie, `mode()` returns several rows.

In [ ]:
# Work with numeric columns only. House_ID is just an identifier, so we exclude it from the statistics.
num = df.select_dtypes(include="number").drop(columns="House_ID")
num.head()

In [ ]:
num.mean()

In [ ]:
num.median()

In [ ]:
num.mode()

## 4. Spread and variability
How far do values stretch around the centre?

- **Range** = max - min
- **Variance** = average squared distance from the mean
- **Standard deviation** = square root of the variance (same units as the data)
- **IQR** = Q3 - Q1, the spread of the middle 50% of the data

In [ ]:
# Range
print(num.max() - num.min())

In [ ]:
# Standard deviation and variance (sample versions, ddof=1, which is the pandas default)
print("Standard deviation:")
print(num.std())
print("\nVariance:")
print(num.var())

In [ ]:
# Variance is the standard deviation squared
import numpy as np
print(np.allclose(num.var(), num.std() ** 2))

In [ ]:
# Manual check on one column. Use len(x) - 1 to match pandas (sample standard deviation).
x = df["Price_USD"]
manual_std = (((x - x.mean()) ** 2).sum() / (len(x) - 1)) ** 0.5
print("Manual :", manual_std)
print("Pandas :", x.std())

In [ ]:
# Deviation of every data point from its column mean
deviations = num - num.mean()
deviations.head()

In [ ]:
# IQR: spread of the middle 50%
Q1 = num.quantile(0.25)
Q3 = num.quantile(0.75)
IQR = Q3 - Q1
print(IQR)

## 5. Shape of the distribution
- **Skewness** ~ 0 means roughly symmetric, > 0 means a longer right tail, < 0 means a longer left tail.
- **Kurtosis** (pandas reports *excess* kurtosis, so a normal distribution = 0). Positive values mean heavier tails and more outliers, and negative values mean lighter tails.

In [ ]:
num.skew()

In [ ]:
num.kurtosis()

In [ ]:
# Compare mean, median and skewness side by side.
# Mean > median usually goes with positive skew, and mean < median with negative skew.
comparison = pd.DataFrame({
    "Mean": num.mean(),
    "Median": num.median(),
    "Skewness": num.skew(),
    "Kurtosis": num.kurtosis(),
})
comparison["Mean - Median"] = comparison["Mean"] - comparison["Median"]
comparison.round(3)

In [ ]:
# One column in detail
col = "Price_USD"
print(f"{col}: mean={df[col].mean():,.0f}  median={df[col].median():,.0f}")
print(f"{col}: skew={df[col].skew():.3f}  kurtosis={df[col].kurt():.3f}")

## 6. Outlier detection (1.5 x IQR rule)
A value is flagged as an outlier if it falls:
- below **Q1 - 1.5 x IQR**, or
- above **Q3 + 1.5 x IQR**

In [ ]:
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

bounds = pd.DataFrame({"Lower bound": lower, "Upper bound": upper})
bounds

In [ ]:
# Count outliers per column
outlier_mask = (num < lower) | (num > upper)
print(outlier_mask.sum())

In [ ]:
# Look at the actual outlier rows for one column (if there are any)
df[outlier_mask["Price_USD"]]

## 7. Visualization: pick the plot for the question

| Plot | Question it answers |
|---|---|
| Histogram | How is one variable distributed? |
| Box plot | How spread out is one variable, and are there outliers? |
| Scatter plot | How are two variables related? |

In [ ]:
plt.hist(df["Price_USD"], bins=20, edgecolor="black")
plt.title("Distribution of House Prices")
plt.xlabel("Price (USD)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.boxplot(df["Price_USD"], vert=False)
plt.title("House Prices: Spread and Outliers")
plt.xlabel("Price (USD)")
plt.show()

In [ ]:
df.plot.scatter(x="Size_sqft", y="Price_USD", alpha=0.6)
plt.title("Price vs Size")
plt.xlabel("Size (sqft)")
plt.ylabel("Price (USD)")
plt.show()

## Key takeaways
- Compare **mean vs median** first. A large gap hints at skew or outliers.
- Use **std / variance** for overall spread and **IQR** when outliers are a concern.
- **Skewness** shows direction and **kurtosis** shows tail weight.
- The **1.5 x IQR rule** is a quick screen. Flagged points are *candidates*, not automatic errors.
- Always visualize: summary numbers can hide the shape of the data.